# Construcción de la base de datos de accidente ofídico, clima y población en Colombia, 2015–2024

Este cuaderno construye, a partir de fuentes públicas, los datos del trabajo sobre la predicción de semanas de riesgo elevado de accidente ofídico. Produce los siguientes archivos:

| Archivo | Contenido |
|---|---|
| `clima_semanal_municipios.csv` | Precipitación (CHIRPS v2.0, suma semanal, mm), temperatura media, máxima y mínima (ERA5-Land, °C) y humedad relativa media (ERA5-Land, %, calculada con la temperatura y el punto de rocío), por municipio y semana. |
| `panel_ofidico_clima_semanal.csv` | Panel municipio-semana con los casos del Sivigila (evento 100), la población proyectada por el DANE y el clima. |
| `dataset_modelado.csv` | Panel con la variable objetivo, la exposición y las variables explicativas (rezagos, acumulados y anomalías), y la partición temporal. |
| `ofidico_municipal.gpkg` y `ofidico_municipal_tabla.csv` | Resumen por municipio y departamento, con tasas suavizadas y estadísticos de autocorrelación espacial, para su cartografía en QGIS. |

**Flujo:** configuración, municipios (GADM), CHIRPS, ERA5-Land, consolidación climática, población y puente DANE–GADM, cruce con el Sivigila, conjunto de modelado y capa espacial.

Las secciones 3 y 4 son reanudables: guardan puntos de control en Google Drive y omiten lo ya procesado. Para procesar otro año basta con cambiar `ANIO_DESCARGA` en la sección 1 y repetir ambas secciones. Las secciones 5 a 9 solo requieren haber ejecutado la sección 1 y que existan los archivos de las secciones anteriores.

## 1. Configuración

In [ ]:
!pip -q install geopandas pyogrio rasterio rasterstats xarray netCDF4 cdsapi requests pandas numpy openpyxl libpysal esda

In [ ]:
import difflib
import glob
import gzip
import os
import re
import unicodedata
import warnings
from datetime import datetime, timedelta

import cdsapi
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests
import xarray as xr
from esda.getisord import G_Local
from esda.moran import Moran, Moran_Local
from google.colab import drive
from libpysal import weights
from rasterio.transform import from_bounds
from rasterstats import zonal_stats

drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Parámetros generales
# ---------------------------------------------------------------------------
# Año que procesan las secciones 3 y 4 (descarga climática).
ANIO_DESCARGA = 2024

# Último año del periodo de entrenamiento. Define la partición temporal del
# conjunto de modelado, la climatología base de las anomalías (sección 8) y
# la separación de casos en la capa espacial (sección 9).
ANIO_FIN_ENTRENAMIENTO = 2021

# Si el inicio de síntomas precede a la notificación en más de este número de
# días, se considera poco confiable y se usa la fecha de notificación.
DIAS_MAX_RETRASO = 30

# ---------------------------------------------------------------------------
# Carpetas y archivos
# ---------------------------------------------------------------------------
DIR_PROYECTO = "/content/drive/MyDrive/proyecto_clima"
DIR_CHIRPS_CP = f"{DIR_PROYECTO}/CHIRPS/checkpoints"
DIR_ERA5_CP = f"{DIR_PROYECTO}/ERA5/checkpoints"
DIR_SALIDA = f"{DIR_PROYECTO}/resultados"
DIR_SIVIGILA = f"{DIR_PROYECTO}/SIVIGILA"
DIR_POBLACION = f"{DIR_PROYECTO}/POBLACION"

# Los temporales de CHIRPS van al disco local de Colab, que es más rápido que Drive.
DIR_CHIRPS_TMP = "/content/temp/chirps"
DIR_ERA5_TMP = f"{DIR_PROYECTO}/ERA5/temp"

CLIMA_CSV = f"{DIR_SALIDA}/clima_semanal_municipios.csv"
POB_CSV = f"{DIR_POBLACION}/poblacion_municipal_anual.csv"
CROSSWALK_CSV = f"{DIR_SALIDA}/crosswalk_dane_gadm_completo.csv"
PANEL_CSV = f"{DIR_SALIDA}/panel_ofidico_clima_semanal.csv"
MODELADO_CSV = f"{DIR_SALIDA}/dataset_modelado.csv"
GPKG = f"{DIR_SALIDA}/ofidico_municipal.gpkg"
TABLA_MUNICIPAL_CSV = f"{DIR_SALIDA}/ofidico_municipal_tabla.csv"

GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_COL_2.json"

for carpeta in [DIR_CHIRPS_CP, DIR_ERA5_CP, DIR_SALIDA, DIR_CHIRPS_TMP, DIR_ERA5_TMP]:
    os.makedirs(carpeta, exist_ok=True)

# ---------------------------------------------------------------------------
# Correcciones manuales del cruce de códigos (se completan tras revisar las
# tablas de unidades sin correspondencia de las secciones 6.2 y 7.2)
# ---------------------------------------------------------------------------
# Código DANE -> municipio_id de GADM, para unidades que el emparejamiento por nombre no resuelve.
MANUAL_MATCH = {
    "25599": "COL.15.74_2",   # Apulo (DANE) = Rafael Reyes (GADM), nombre anterior del municipio
}

# Códigos que el Sivigila usa pero que no figuran en la lista del DANE.
MANUAL_CASOS = {
    "27086": "COL.13.7_2",    # Belén de Bajirá (Chocó)
    "27493": "COL.13.7_2",    # el Sivigila también registra Belén de Bajirá con este código
}

COLS_UNIDAD = ["municipio_id", "departamento", "municipio", "tipo_administrativo"]


def inicios_de_semana(anio):
    """Inicios de bloque de 7 días, contados desde el 1 de enero, que caen dentro del año."""
    actual, fin = datetime(anio, 1, 1), datetime(anio + 1, 1, 1)
    semanas = []
    while actual < fin:
        semanas.append(actual)
        actual += timedelta(days=7)
    return semanas


def cargar_gadm():
    """Unidades de segundo nivel de GADM 4.1 con columnas en español y CRS EPSG:4326."""
    return (
        gpd.read_file(GADM_URL)[["GID_2", "NAME_1", "NAME_2", "TYPE_2", "geometry"]]
        .rename(columns={"GID_2": "municipio_id", "NAME_1": "departamento",
                         "NAME_2": "municipio", "TYPE_2": "tipo_administrativo"})
        .to_crs("EPSG:4326")
        .copy()
    )

## 2. Municipios de Colombia (GADM 4.1, nivel 2)

Se conservan las 1.119 unidades de segundo nivel, con su tipo administrativo (`tipo_administrativo`), para poder filtrar después si se requiere (por ejemplo, solo `Municipio`).

In [ ]:
municipios = cargar_gadm()

assert municipios["municipio_id"].is_unique, "Hay municipio_id duplicados"
assert not municipios.geometry.is_empty.any(), "Hay geometrías vacías"
assert municipios.geometry.is_valid.all(), "Hay geometrías inválidas"

print(f"Unidades cargadas: {len(municipios)} | CRS: {municipios.crs}")
print(municipios["tipo_administrativo"].value_counts())

## 3. CHIRPS: precipitación semanal

Para cada semana se descargan los siete ráster diarios, se suman, se calcula el promedio por municipio con `zonal_stats` y se guarda un CSV de control en `CHIRPS/checkpoints/chirps_AAAA-MM-DD.csv`.

In [ ]:
URL_CHIRPS = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/tifs/p05"


def descargar_chirps_dia(fecha):
    """Descarga el ráster diario de CHIRPS y devuelve la ruta del GeoTIFF descomprimido."""
    nombre = f"chirps-v2.0.{fecha:%Y.%m.%d}.tif"
    url = f"{URL_CHIRPS}/{fecha.year}/{nombre}.gz"
    ruta_gz = os.path.join(DIR_CHIRPS_TMP, f"{nombre}.gz")
    ruta_tif = os.path.join(DIR_CHIRPS_TMP, nombre)

    if os.path.exists(ruta_tif):
        return ruta_tif

    respuesta = requests.get(url, timeout=120)
    if respuesta.status_code != 200:
        raise RuntimeError(f"CHIRPS {fecha:%Y-%m-%d}: HTTP {respuesta.status_code}")

    with open(ruta_gz, "wb") as f:
        f.write(respuesta.content)
    with gzip.open(ruta_gz, "rb") as f_in, open(ruta_tif, "wb") as f_out:
        f_out.write(f_in.read())
    os.remove(ruta_gz)
    return ruta_tif


def crear_chirps_semanal(inicio):
    """Suma siete días de CHIRPS y guarda el GeoTIFF semanal; devuelve su ruta."""
    matrices, perfil = [], None

    for i in range(7):
        ruta = descargar_chirps_dia(inicio + timedelta(days=i))
        with rasterio.open(ruta) as src:
            datos = src.read(1).astype("float32")
            datos[datos < 0] = np.nan   # CHIRPS codifica los datos faltantes con valores negativos
            matrices.append(datos)
            if perfil is None:
                perfil = src.profile.copy()

    pila = np.stack(matrices)
    semanal = np.nansum(pila, axis=0)
    semanal[np.all(np.isnan(pila), axis=0)] = np.nan   # siete días sin dato: la celda queda sin dato

    ruta_salida = os.path.join(DIR_CHIRPS_TMP, f"weekly_{inicio:%Y%m%d}.tif")
    perfil.update(dtype="float32", count=1, nodata=-9999, compress="deflate",
                  tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(ruta_salida, "w", **perfil) as dst:
        dst.write(np.where(np.isnan(semanal), -9999, semanal).astype("float32"), 1)
    return ruta_salida


def precipitacion_semanal(inicio):
    """Precipitación semanal media por municipio."""
    ruta = crear_chirps_semanal(inicio)
    estadisticos = zonal_stats(municipios, ruta, stats=["mean"], nodata=-9999, all_touched=True)

    resultado = municipios[COLS_UNIDAD].copy()
    resultado["week_start"] = inicio.strftime("%Y-%m-%d")
    resultado["precip_mm"] = [e["mean"] if e["mean"] is not None else np.nan for e in estadisticos]
    return resultado


def procesar_semana_chirps(inicio):
    """Procesa una semana y guarda su archivo de control; no hace nada si ya existe."""
    semana = inicio.strftime("%Y-%m-%d")
    control = os.path.join(DIR_CHIRPS_CP, f"chirps_{semana}.csv")
    if os.path.exists(control):
        return None

    resultado = precipitacion_semanal(inicio)

    if len(resultado) != len(municipios) or resultado["municipio_id"].nunique() != len(municipios):
        raise ValueError(f"{semana}: se esperaban {len(municipios)} unidades únicas, "
                         f"se obtuvieron {len(resultado)}")

    n_sin_dato = resultado["precip_mm"].isna().sum()
    if n_sin_dato:
        print(f"   Aviso {semana}: {n_sin_dato} unidades sin precipitación válida")

    resultado.to_csv(control, index=False)

    # Se borran los ráster temporales (diarios y semanal) para no llenar el disco.
    for ruta in glob.glob(os.path.join(DIR_CHIRPS_TMP, "*.tif")):
        os.remove(ruta)
    return resultado

In [ ]:
# Ejecución de CHIRPS (reanudable)
semanas = inicios_de_semana(ANIO_DESCARGA)
pendientes = [s for s in semanas
              if not os.path.exists(os.path.join(DIR_CHIRPS_CP, f"chirps_{s:%Y-%m-%d}.csv"))]

print(f"CHIRPS {ANIO_DESCARGA}: {len(semanas)} semanas, "
      f"{len(semanas) - len(pendientes)} ya procesadas, {len(pendientes)} pendientes")

for i, semana in enumerate(pendientes, start=1):
    print(f"  {semana:%Y-%m-%d} ({i}/{len(pendientes)})")
    procesar_semana_chirps(semana)

print("CHIRPS finalizado")

## 4. ERA5-Land: temperatura y humedad relativa semanales

Para cada semana se descargan 168 horas de `2m_temperature` y `2m_dewpoint_temperature` desde el Climate Data Store (las semanas que cruzan de mes se descargan por partes). Se calculan la temperatura media, máxima y mínima y la humedad relativa media con la fórmula de Magnus, y se promedian por municipio.

El punto de control es un único CSV por año, `ERA5/checkpoints/era5_semanal_AAAA.csv`. Se requiere el secreto `CDS_API_KEY` en Colab.

In [ ]:
def crear_cliente_cds():
    """Crea el cliente del CDS con el secreto de Colab, o con ~/.cdsapirc si ya existe."""
    ruta_rc = os.path.expanduser("~/.cdsapirc")

    try:
        from google.colab import userdata
        clave = userdata.get("CDS_API_KEY")
    except Exception:
        clave = None

    if clave:
        with open(ruta_rc, "w") as f:
            f.write("url: https://cds.climate.copernicus.eu/api\n")
            f.write(f"key: {clave}\n")
    elif not os.path.exists(ruta_rc):
        raise RuntimeError("No se encontró CDS_API_KEY en los secretos de Colab ni un ~/.cdsapirc existente.")

    return cdsapi.Client()


cliente_cds = crear_cliente_cds()
print("Cliente CDS listo")

In [ ]:
# Caja que cubre Colombia: [norte, oeste, sur, este]
AREA_ERA5 = [13.6, -79.6, -4.6, -66.4]


def solicitar_era5(anio, mes, dias, destino):
    """Descarga las horas de t2m y d2m de los días indicados de un mes."""
    cliente_cds.retrieve(
        "reanalysis-era5-land",
        {
            "variable": ["2m_temperature", "2m_dewpoint_temperature"],
            "year": str(anio),
            "month": f"{mes:02d}",
            "day": [f"{d:02d}" for d in dias],
            "time": [f"{h:02d}:00" for h in range(24)],
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": AREA_ERA5,
        },
        destino,
    )


def descargar_era5_semana(inicio, ruta_nc):
    """Descarga una semana de ERA5-Land; si cruza de mes, descarga por partes y las une."""
    inicio = pd.Timestamp(inicio)

    grupos = {}   # (año, mes) -> días de la semana en ese mes
    for i in range(7):
        d = inicio + pd.Timedelta(days=i)
        grupos.setdefault((d.year, d.month), []).append(d.day)

    if len(grupos) == 1:
        (anio, mes), dias = next(iter(grupos.items()))
        print(f"   Descargando {anio}-{mes:02d}")
        solicitar_era5(anio, mes, dias, ruta_nc)
        return

    partes = []
    for k, ((anio, mes), dias) in enumerate(grupos.items()):
        ruta_parte = os.path.join(DIR_ERA5_TMP, f"part_{inicio:%Y%m%d}_{k}.nc")
        print(f"   Descargando {anio}-{mes:02d}")
        solicitar_era5(anio, mes, dias, ruta_parte)
        partes.append(ruta_parte)

    conjuntos = [xr.open_dataset(p) for p in partes]
    unido = xr.concat(conjuntos, dim="valid_time")
    unido.to_netcdf(ruta_nc)

    unido.close()
    for ds_parte in conjuntos:
        ds_parte.close()
    for p in partes:
        if os.path.exists(p):
            os.remove(p)


def guardar_tif(datos, ruta_salida):
    """Guarda un DataArray 2D (latitude, longitude) como GeoTIFF en EPSG:4326."""
    lats, lons = datos.latitude.values, datos.longitude.values
    res_x, res_y = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])

    transformacion = from_bounds(
        lons.min() - res_x / 2, lats.min() - res_y / 2,
        lons.max() + res_x / 2, lats.max() + res_y / 2,
        len(lons), len(lats),
    )

    valores = datos.values
    if lats[0] < lats[-1]:   # rasterio espera las filas ordenadas de norte a sur
        valores = np.flipud(valores)

    with rasterio.open(
        ruta_salida, "w", driver="GTiff",
        height=valores.shape[0], width=valores.shape[1], count=1,
        dtype="float32", crs="EPSG:4326", transform=transformacion, nodata=-9999,
    ) as dst:
        dst.write(valores.astype("float32"), 1)


def clima_era5_semanal(inicio):
    """Descarga, si hace falta, y calcula las variables ERA5 de una semana por municipio."""
    inicio = pd.Timestamp(inicio)
    semana = inicio.strftime("%Y-%m-%d")

    ruta_nc = os.path.join(DIR_ERA5_TMP, f"era5_{inicio:%Y%m%d}.nc")
    if not os.path.exists(ruta_nc):
        descargar_era5_semana(inicio, ruta_nc)
    else:
        print("   NetCDF existente, se reutiliza")

    ds = xr.open_dataset(ruta_nc)
    rasters = {}

    try:
        if len(ds.valid_time) != 168:
            raise ValueError(f"{semana}: se esperaban 168 horas y hay {len(ds.valid_time)}")

        t2m_c = ds["t2m"] - 273.15   # de kelvin a grados Celsius
        d2m_c = ds["d2m"] - 273.15

        # Humedad relativa: cociente entre la presión de vapor real (punto de rocío)
        # y la de saturación (temperatura del aire), con la aproximación de Magnus.
        p_sat = 6.112 * np.exp((17.625 * t2m_c) / (243.04 + t2m_c))
        p_vap = 6.112 * np.exp((17.625 * d2m_c) / (243.04 + d2m_c))
        rh = (100 * (p_vap / p_sat)).clip(min=0, max=100)

        variables = {
            "temp_mean": t2m_c.mean(dim="valid_time"),
            "temp_max": t2m_c.max(dim="valid_time"),
            "temp_min": t2m_c.min(dim="valid_time"),
            "rh_mean": rh.mean(dim="valid_time"),
        }
        for nombre, datos in variables.items():
            ruta_tif = os.path.join(DIR_ERA5_TMP, f"{nombre}_{inicio:%Y%m%d}.tif")
            guardar_tif(datos, ruta_tif)
            rasters[nombre] = ruta_tif

        resultado = municipios[COLS_UNIDAD].copy()
        resultado["week_start"] = semana
        for nombre, ruta_tif in rasters.items():
            estadisticos = zonal_stats(municipios, ruta_tif, stats=["mean"], nodata=-9999, all_touched=True)
            resultado[nombre] = [e["mean"] if e["mean"] is not None else np.nan for e in estadisticos]

        if len(resultado) != len(municipios) or resultado["municipio_id"].nunique() != len(municipios):
            raise ValueError(f"{semana}: resultado con municipios faltantes o duplicados")
        return resultado

    finally:
        ds.close()
        for ruta_tif in rasters.values():
            if os.path.exists(ruta_tif):
                os.remove(ruta_tif)

In [ ]:
# Ejecución de ERA5-Land (reanudable; un CSV por año)
ERA5_ANUAL_CSV = os.path.join(DIR_ERA5_CP, f"era5_semanal_{ANIO_DESCARGA}.csv")

era5_anual = pd.DataFrame()
if os.path.exists(ERA5_ANUAL_CSV):
    try:
        era5_anual = pd.read_csv(ERA5_ANUAL_CSV)
    except Exception:
        era5_anual = pd.DataFrame()

completadas = set(era5_anual["week_start"].astype(str)) if "week_start" in era5_anual.columns else set()

semanas = inicios_de_semana(ANIO_DESCARGA)
print(f"ERA5-Land {ANIO_DESCARGA}: {len(semanas)} semanas, {len(completadas)} ya procesadas, "
      f"{len(municipios)} municipios")

for inicio in semanas:
    semana = inicio.strftime("%Y-%m-%d")
    if semana in completadas:
        continue

    print(f"  {semana}")
    try:
        resultado = clima_era5_semanal(inicio)
    except Exception as error:
        print(f"Error en {semana}: {error!r}")
        print("Se detiene el proceso; las semanas anteriores ya están guardadas.")
        break

    era5_anual = pd.concat([era5_anual, resultado], ignore_index=True)
    era5_anual.to_csv(ERA5_ANUAL_CSV, index=False)
    completadas.add(semana)
    print(f"   Punto de control guardado ({len(era5_anual)} filas acumuladas)")

    ruta_nc = os.path.join(DIR_ERA5_TMP, f"era5_{inicio:%Y%m%d}.nc")
    if os.path.exists(ruta_nc):
        os.remove(ruta_nc)

print("ERA5-Land finalizado")
if len(era5_anual):
    print(era5_anual[["temp_mean", "temp_max", "temp_min", "rh_mean"]].isna().sum())

## 5. Consolidación de CHIRPS y ERA5-Land

Esta sección es independiente de las anteriores: solo necesita Drive montado y las rutas de la sección 1. Une los puntos de control de CHIRPS (uno por semana) y de ERA5-Land (uno por año) en una tabla con llave `municipio_id` y `week_start`, y la guarda en `resultados/clima_semanal_municipios.csv`.

In [ ]:
def leer_checkpoints(carpeta, patron):
    archivos = sorted(glob.glob(os.path.join(carpeta, patron)))
    if not archivos:
        raise FileNotFoundError(f"No hay archivos '{patron}' en {carpeta}. ¿Está Drive montado?")
    print(f"{len(archivos)} archivos en {os.path.basename(os.path.dirname(carpeta))}")
    return pd.concat([pd.read_csv(a, dtype={"week_start": str}) for a in archivos], ignore_index=True)


LLAVE = ["municipio_id", "week_start"]
VARS_ERA5 = ["temp_mean", "temp_max", "temp_min", "rh_mean"]

chirps = leer_checkpoints(DIR_CHIRPS_CP, "chirps_*.csv")
era5 = leer_checkpoints(DIR_ERA5_CP, "era5_semanal_*.csv")

# Una semana puede estar en más de un archivo; se conserva la primera aparición.
for nombre, tabla_fuente in [("CHIRPS", chirps), ("ERA5", era5)]:
    n_dup = tabla_fuente.duplicated(LLAVE).sum()
    if n_dup:
        print(f"Aviso {nombre}: {n_dup} filas duplicadas; se conserva la primera")
chirps = chirps.drop_duplicates(LLAVE)
era5 = era5.drop_duplicates(LLAVE)

clima = chirps[LLAVE + ["precip_mm"]].merge(era5[LLAVE + VARS_ERA5], on=LLAVE, how="outer", indicator=True)

print("\nCobertura del cruce:")
print(clima["_merge"].map({"both": "en ambos", "left_only": "solo CHIRPS",
                           "right_only": "solo ERA5"}).value_counts())
clima = clima.drop(columns="_merge")

# Los metadatos del municipio vienen en los propios archivos de control de CHIRPS.
unidades_clima = chirps[COLS_UNIDAD].drop_duplicates("municipio_id")
clima = clima.merge(unidades_clima, on="municipio_id", how="left")
clima["week_start"] = pd.to_datetime(clima["week_start"])
clima = (clima[COLS_UNIDAD + ["week_start", "precip_mm"] + VARS_ERA5]
         .sort_values(["municipio_id", "week_start"]).reset_index(drop=True))

print(f"\nFilas: {len(clima):,} | municipios: {clima['municipio_id'].nunique()} | "
      f"semanas: {clima['week_start'].nunique()}")
print(f"Rango: {clima['week_start'].min():%Y-%m-%d} a {clima['week_start'].max():%Y-%m-%d}")
print("\nValores faltantes:")
print(clima[["precip_mm"] + VARS_ERA5].isna().sum())

clima.to_csv(CLIMA_CSV, index=False)
print("\nGuardado:", CLIMA_CSV)

## 6. Población (DANE) y puente de códigos DANE–GADM

Esta sección tiene dos objetivos, ambos necesarios para pasar de casos a incidencia:

1. Descargar la población municipal oficial del DANE.
2. Construir el puente de códigos. El Sivigila y el DANE identifican los municipios con el código DANE de cinco dígitos, pero GADM no lo incluye (la columna `CC_2` viene vacía). El puente se obtiene emparejando los nombres oficiales del DANE con los de GADM dentro de cada departamento, y lo utilizan tanto la población como los casos.

**Fuente.** DANE, *Proyecciones y retroproyecciones de población municipal por área*, basadas en el Censo Nacional de Población y Vivienda 2018 y su actualización posterior a la pandemia de COVID-19. Se usan dos archivos con la misma base censal, por lo que no hay quiebre de la serie entre 2017 y 2018:

- Retroproyección 2005–2017 (`DCD-area-proypoblacion-Mun-2005-2017_VP.xlsx`).
- Proyección 2018–2042 (`PPED-AreaMun-2018-2042_VP.xlsx`).

**Variables.** Población total y rural (centros poblados y rural disperso). La población rural es la más pertinente como exposición, porque la incidencia del accidente ofídico se calcula sobre la población en riesgo, principalmente rural (Chaves et al., 2015). Los valores son anuales (referidos a mitad de año, como en las proyecciones del DANE) y se repiten en todas las semanas del año.

Si la descarga automática falla, los dos archivos pueden bajarse de la página de proyecciones del DANE y guardarse en `proyecto_clima/POBLACION/`.

### 6.0 Funciones auxiliares para el cruce de nombres

In [ ]:
# Departamentos del DANE: código -> nombre normalizado (sin tildes, espacios ni símbolos).
DEPARTAMENTOS_DANE = {
    5: "ANTIOQUIA", 8: "ATLANTICO", 11: "BOGOTADC", 13: "BOLIVAR", 15: "BOYACA",
    17: "CALDAS", 18: "CAQUETA", 19: "CAUCA", 20: "CESAR", 23: "CORDOBA",
    25: "CUNDINAMARCA", 27: "CHOCO", 41: "HUILA", 44: "LAGUAJIRA", 47: "MAGDALENA",
    50: "META", 52: "NARINO", 54: "NORTEDESANTANDER", 63: "QUINDIO", 66: "RISARALDA",
    68: "SANTANDER", 70: "SUCRE", 73: "TOLIMA", 76: "VALLEDELCAUCA", 81: "ARAUCA",
    85: "CASANARE", 86: "PUTUMAYO", 88: "SANANDRES", 91: "AMAZONAS", 94: "GUAINIA",
    95: "GUAVIARE", 97: "VAUPES", 99: "VICHADA",
}


def norm(texto):
    """Mayúsculas sin tildes, paréntesis, espacios ni símbolos: 'Mirití-Paraná (CD)' -> 'MIRITIPARANA'."""
    texto = re.sub(r"\(.*?\)", "", str(texto))
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^A-Z0-9]", "", texto.upper())


def codigo_departamento(nombre_gadm):
    """Código DANE del departamento a partir del nombre que trae GADM."""
    n = norm(nombre_gadm)
    for codigo, canonico in DEPARTAMENTOS_DANE.items():
        if n == canonico:
            return codigo
    candidatos = [c for c, canon in DEPARTAMENTOS_DANE.items() if canon in n or n in canon]
    return candidatos[0] if len(candidatos) == 1 else None


def preparar_gadm(df_clima):
    """Unidades GADM únicas del panel climático, con código DANE de departamento y nombre normalizado."""
    gadm = df_clima[COLS_UNIDAD].drop_duplicates("municipio_id").reset_index(drop=True)
    codigos = {d: codigo_departamento(d) for d in gadm["departamento"].unique()}
    sin_codigo = [d for d, c in codigos.items() if c is None]
    if sin_codigo:
        raise ValueError(f"Departamentos de GADM sin código DANE: {sin_codigo}. Ajustar DEPARTAMENTOS_DANE.")
    gadm["dpto"] = gadm["departamento"].map(codigos)
    gadm["clave"] = gadm["municipio"].map(norm)
    return gadm


def construir_crosswalk(unidades, gadm, manual, peso_col="casos"):
    """
    Empareja cada código DANE con un municipio_id de GADM dentro de su departamento.

    `unidades` debe traer dane5, dpto, nombre y la columna de peso (casos o población).
    Se procesan primero las unidades de mayor peso, para que en un conflicto de nombres
    conserve la pareja la unidad más grande. El emparejamiento va de lo más a lo menos
    seguro: corrección manual, nombre idéntico, un nombre contenido en el otro y, por
    último, similitud de texto (razón de difflib >= 0,85 con margen sobre el segundo candidato).
    """
    unidades = unidades.assign(clave=unidades["nombre"].map(norm)).sort_values(peso_col, ascending=False)
    filas, usados = [], set()
    nombre_gadm = gadm.set_index("municipio_id")["municipio"].to_dict()

    def registrar(u, municipio_id, metodo):
        filas.append({"dane5": u.dane5, "municipio_id": municipio_id, "metodo": metodo,
                      "nombre_dane": u.nombre, "nombre_gadm": nombre_gadm[municipio_id],
                      peso_col: getattr(u, peso_col)})
        usados.add(municipio_id)

    # 1) Correcciones manuales. Si apuntan a un municipio_id inexistente se ignoran, con aviso.
    for u in unidades.itertuples():
        if u.dane5 in manual:
            if manual[u.dane5] in nombre_gadm:
                registrar(u, manual[u.dane5], "manual")
            else:
                print(f"Aviso: MANUAL_MATCH {u.dane5} apunta a '{manual[u.dane5]}', que no existe en GADM; se ignora.")

    # 2) Nombre idéntico (normalizado) dentro del departamento, si es único en GADM.
    exacto = {k: v[0] for k, v in gadm.groupby(["dpto", "clave"])["municipio_id"].apply(list).items()
              if len(v) == 1}
    hechos = {f["dane5"] for f in filas}
    for u in unidades.itertuples():
        mid = exacto.get((u.dpto, u.clave))
        if u.dane5 not in hechos and mid is not None and mid not in usados:
            registrar(u, mid, "exacto")
            hechos.add(u.dane5)

    # 3) Un nombre contiene al otro, o son muy parecidos. Estos cruces se revisan a mano.
    for u in unidades.itertuples():
        if u.dane5 in hechos:
            continue
        candidatos = gadm[(gadm["dpto"] == u.dpto) & ~gadm["municipio_id"].isin(usados)]
        contiene = [r.municipio_id for r in candidatos.itertuples()
                    if min(len(r.clave), len(u.clave)) >= 4 and (r.clave in u.clave or u.clave in r.clave)]
        if len(contiene) == 1:
            registrar(u, contiene[0], "contiene")
            hechos.add(u.dane5)
            continue
        puntajes = sorted(
            [(difflib.SequenceMatcher(None, u.clave, r.clave).ratio(), r.municipio_id)
             for r in candidatos.itertuples()],
            reverse=True,
        )
        puntajes = [p for p in puntajes if p[0] >= 0.85]
        if len(puntajes) == 1 or (len(puntajes) > 1 and puntajes[0][0] - puntajes[1][0] >= 0.05):
            registrar(u, puntajes[0][1], "similar")
            hechos.add(u.dane5)

    return pd.DataFrame(filas)


def reporte_crosswalk(crosswalk, unidades, peso_col="casos", etiqueta="casos"):
    """Imprime la cobertura del cruce y las tablas que requieren revisión manual."""
    sin_match = unidades[~unidades["dane5"].isin(crosswalk["dane5"])]
    total = unidades[peso_col].sum()
    print(f"Método de cruce (unidades DANE y {etiqueta}):")
    print(crosswalk.groupby("metodo").agg(unidades=("dane5", "size"), **{etiqueta: (peso_col, "sum")}).to_string())
    print(f"\nSin correspondencia: {len(sin_match)} unidades, {sin_match[peso_col].sum():,.0f} {etiqueta} "
          f"({sin_match[peso_col].sum() / total:.2%} del total)")

    print("\nCruces no exactos, a verificar (deben ser el mismo municipio):")
    print(crosswalk[crosswalk["metodo"].isin(["contiene", "similar"])]
          [["dane5", "nombre_dane", "nombre_gadm", "metodo", peso_col]].to_string(index=False))

    print(f"\nSin correspondencia, de mayor a menor {etiqueta}:")
    print(sin_match[["dane5", "nombre", peso_col]].head(30).to_string(index=False))
    return sin_match

### 6.1 Descarga y lectura de la población

In [ ]:
# Fuentes oficiales: población municipal por área (Censo 2018, actualización posterior a la COVID-19).
# https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion
URL_DANE = "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal"

FUENTES_POBLACION = {   # etiqueta: (archivo, años que se toman de él)
    "retroproyección": (f"{URL_DANE}/DCD-area-proypoblacion-Mun-2005-2017_VP.xlsx", range(2005, 2018)),
    "proyección": (f"{URL_DANE}/PPED-AreaMun-2018-2042_VP.xlsx", range(2018, 2043)),
}


def descargar_dane(url, carpeta):
    """Descarga el Excel del DANE a Drive; si ya está, lo reutiliza."""
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, os.path.basename(url))
    if os.path.exists(destino) and os.path.getsize(destino) > 10_000:
        return destino

    respuesta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=300)
    if respuesta.status_code != 200 or not respuesta.content.startswith(b"PK"):
        raise RuntimeError(
            f"No se pudo descargar {url} (HTTP {respuesta.status_code}). "
            f"Descargar el archivo desde la página de proyecciones del DANE y guardarlo en {carpeta}."
        )
    with open(destino, "wb") as f:
        f.write(respuesta.content)
    return destino


# Nombres posibles de cada columna, ya normalizados con norm(); los archivos del DANE
# han cambiado de encabezados entre versiones.
COL_CODIGO = ("MPIO", "CODMPIO", "CODIGOMUNICIPIO", "CODIGODANE", "DIVIPOLA", "CODIGO")
COL_NOMBRE = ("DPMP", "MUNICIPIO", "NOMBREMUNICIPIO", "NOMMPIO", "NOMBRE")
COL_AREA = ("AREAGEOGRAFICA", "AREA")
COL_ANIO = ("ANO", "ANIO", "YEAR")
COL_POBL = ("POBLACION", "POB", "TOTALPOBLACION", "TOTAL", "POBLACIONTOTAL", "VALOR")
COL_DPTO = ("DP", "DPTO", "CODDPTO", "CODDEPARTAMENTO", "CODIGODEPARTAMENTO", "CODDP", "COD_DPTO")


def _nombre_col(valor):
    """Nombre de columna normalizado; los años numéricos (2005 o 2005.0) quedan como '2005'."""
    if pd.isna(valor):
        return ""
    if isinstance(valor, (int, float, np.number)) and float(valor).is_integer():
        return str(int(valor))
    return norm(valor)


def _a_numero(serie):
    """Convierte a número; tolera espacios y separadores de miles (1.234.567 o 1,234,567)."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    t = serie.astype(str).str.replace("\u00a0", "", regex=False).str.strip()
    t = t.str.replace(r"^(\d{1,3})((?:[.,]\d{3})+)$",
                      lambda m: m.group(1) + re.sub(r"[.,]", "", m.group(2)), regex=True)
    return pd.to_numeric(t, errors="coerce")


def _es_anio(nombre):
    m = re.fullmatch(r"(?:POBLACION|POB)?((?:19|20)\d\d)", nombre)
    return int(m.group(1)) if m else None


def _tipo_area(texto):
    n = norm(texto)
    if n.startswith("TOTAL"):
        return "total"
    if n.startswith("CABECERA"):
        return "cabecera"
    return "rural"   # "Centros Poblados y Rural Disperso" o "Resto"


def leer_poblacion(ruta):
    """
    Lee un Excel de población municipal del DANE, en formato largo o ancho.
    Devuelve las columnas dane5, nombre, anio, total, cabecera y rural.
    """
    xls = pd.ExcelFile(ruta)
    partes = []

    for hoja in xls.sheet_names:
        bruto = xls.parse(hoja, header=None)

        # Fila de encabezado: la que tiene una columna de código y una de año (o columnas por año).
        fila = None
        for i in range(min(40, len(bruto))):
            nombres = [_nombre_col(c) for c in bruto.iloc[i]]
            if any(n in COL_CODIGO for n in nombres) and (
                any(n in COL_ANIO for n in nombres) or any(_es_anio(n) for n in nombres)
            ):
                fila = i
                break
        if fila is None:
            continue

        vistos, cols = {}, []
        for n in nombres:
            vistos[n] = vistos.get(n, 0) + 1
            cols.append(n if vistos[n] == 1 else f"{n}_{vistos[n]}")
        df = bruto.iloc[fila + 1:].copy()
        df.columns = cols

        def buscar(opciones):
            return next((c for c in cols if c in opciones), None)

        c_cod, c_nom, c_area = buscar(COL_CODIGO), buscar(COL_NOMBRE), buscar(COL_AREA)
        c_anio, c_pob = buscar(COL_ANIO), buscar(COL_POBL)
        c_dp = buscar(COL_DPTO)
        if c_dp is None:
            # Respaldo: columna numérica con valores de 1 a 99 (código de departamento).
            for c in cols:
                if c in (c_cod, c_anio, c_pob) or _es_anio(c):
                    continue
                v = pd.to_numeric(df[c], errors="coerce").dropna()
                if len(v) > 0.9 * len(df) and v.between(1, 99).all():
                    c_dp = c
                    break

        if c_anio and c_pob:   # formato largo
            largo = pd.DataFrame({
                "codigo": df[c_cod], "anio": df[c_anio], "poblacion": df[c_pob],
                "area": df[c_area] if c_area else "Total",
                "nombre": df[c_nom] if c_nom else None,
                "dp": df[c_dp] if c_dp else np.nan,
            })
        else:   # formato ancho: una columna por año
            col_anios = {c: _es_anio(c) for c in cols if _es_anio(c)}
            if not col_anios:
                raise ValueError(
                    f"{os.path.basename(ruta)} (hoja '{hoja}'): no se identificó la columna de población "
                    f"ni columnas por año. Columnas encontradas: {cols}"
                )
            base = pd.DataFrame({
                "codigo": df[c_cod],
                "area": df[c_area] if c_area else "Total",
                "nombre": df[c_nom] if c_nom else None,
                "dp": df[c_dp] if c_dp else np.nan,
            })
            ancho_df = pd.concat([base, df[list(col_anios)]], axis=1)
            largo = ancho_df.melt(id_vars=list(base.columns), value_vars=list(col_anios),
                                  var_name="anio", value_name="poblacion")
            largo["anio"] = largo["anio"].map(col_anios)
        largo["area"] = largo["area"].map(_tipo_area)

        largo["codigo"] = _a_numero(largo["codigo"])
        largo["dp"] = _a_numero(largo["dp"])
        largo["anio"] = _a_numero(largo["anio"])
        largo["poblacion"] = _a_numero(largo["poblacion"])
        n0 = len(largo)
        n_sin_codigo, n_sin_anio, n_sin_pob = (int(largo[c].isna().sum()) for c in ("codigo", "anio", "poblacion"))
        largo = largo.dropna(subset=["codigo", "anio", "poblacion"])

        # Si el código viene sin departamento (1 a 999), se compone con el código del departamento.
        sin_dp = largo["codigo"] < 1000
        largo.loc[sin_dp, "codigo"] = largo.loc[sin_dp, "dp"].fillna(0) * 1000 + largo.loc[sin_dp, "codigo"]
        n_codigo_corto = int((largo["codigo"] < 1000).sum())
        largo = largo[largo["codigo"] >= 1000]
        print(f"   [{os.path.basename(ruta)} | hoja '{hoja}'] encabezado en la fila {fila + 1}; "
              f"columnas: {cols[:10]}{'...' if len(cols) > 10 else ''}")
        print(f"   filas leídas: {n0:,} | sin código: {n_sin_codigo:,} | sin año: {n_sin_anio:,} | "
              f"sin población: {n_sin_pob:,} | código sin departamento: {n_codigo_corto:,} | válidas: {len(largo):,}")
        largo["dane5"] = largo["codigo"].astype(int).astype(str).str.zfill(5)
        largo["anio"] = largo["anio"].astype(int)
        partes.append(largo[["dane5", "nombre", "anio", "area", "poblacion"]])

    if not partes:
        raise ValueError(
            f"No se encontró la tabla de población en {os.path.basename(ruta)} (hojas: {xls.sheet_names}). "
            "Revisar los nombres de las columnas del encabezado."
        )

    largo = pd.concat(partes, ignore_index=True)
    nombres = largo.dropna(subset=["nombre"]).drop_duplicates("dane5").set_index("dane5")["nombre"]
    ancho_df = largo.pivot_table(index=["dane5", "anio"], columns="area", values="poblacion", aggfunc="first")
    for col in ("total", "cabecera", "rural"):
        if col not in ancho_df.columns:
            ancho_df[col] = np.nan
    ancho_df["total"] = ancho_df["total"].fillna(ancho_df["cabecera"] + ancho_df["rural"])
    ancho_df = ancho_df.reset_index()
    ancho_df["nombre"] = ancho_df["dane5"].map(nombres)
    return ancho_df[["dane5", "nombre", "anio", "total", "cabecera", "rural"]]


# Descarga y lectura: retroproyección 2005-2017 y proyección 2018-2042 (misma base censal).
tablas = []
for etiqueta, (url, anios_fuente) in FUENTES_POBLACION.items():
    ruta = descargar_dane(url, DIR_POBLACION)
    t_total = leer_poblacion(ruta)
    t = t_total[t_total["anio"].isin(anios_fuente)]
    if t.empty:
        raise RuntimeError(
            f"{os.path.basename(ruta)}: no quedó ningún municipio en los años {min(anios_fuente)}-{max(anios_fuente)}. "
            f"Años leídos: {[int(a) for a in sorted(t_total['anio'].unique())[:10]]}. "
            "Revisar el diagnóstico de columnas y filas descartadas impreso arriba."
        )
    print(f"{etiqueta}: {t['dane5'].nunique()} municipios, años {t['anio'].min()}-{t['anio'].max()}")
    tablas.append(t)

pob = pd.concat(tablas, ignore_index=True)
nombres_dane = pob.dropna(subset=["nombre"]).drop_duplicates("dane5").set_index("dane5")["nombre"]
pob["nombre"] = pob["dane5"].map(nombres_dane)

# Control: un salto brusco del total nacional entre 2017 y 2018 delataría un quiebre entre los dos archivos.
nacional = pob.groupby("anio")["total"].sum()
control = pd.DataFrame({"poblacion_total": nacional, "variacion_%": (nacional.pct_change() * 100).round(2)})
print("\nPoblación total (suma de municipios) por año:")
print(control.loc[2014:2025].to_string())

pob.to_csv(POB_CSV, index=False)
print("\nGuardado:", POB_CSV)

### 6.2 Puente DANE–GADM

La celda imprime los cruces no exactos, que deben corresponder al mismo municipio, y las unidades sin correspondencia. Para corregir un caso se agrega a `MANUAL_MATCH` (sección 1, con la forma `{"código DANE": "municipio_id de GADM"}`) y se vuelve a ejecutar desde la sección 6.0. Las unidades del DANE sin pareja en GADM suelen ser municipios creados después de 2005 que GADM no separa; sus casos se excluyen y se reportan en la sección 7.2.

In [ ]:
# Puente completo: todos los municipios del DANE.
df_clima = pd.read_csv(CLIMA_CSV, parse_dates=["week_start"])
gadm = preparar_gadm(df_clima)

unidades_pob = (
    pob.groupby("dane5")
    .agg(nombre=("nombre", "first"), poblacion=("total", "mean"))
    .reset_index()
)
unidades_pob["dpto"] = unidades_pob["dane5"].str[:2].astype(int)
if unidades_pob["nombre"].isna().any():
    raise ValueError("La tabla del DANE no trae nombres de municipio; no es posible el cruce con GADM.")

crosswalk_pob = construir_crosswalk(unidades_pob, gadm, MANUAL_MATCH, peso_col="poblacion")
sin_match_pob = reporte_crosswalk(crosswalk_pob, unidades_pob, peso_col="poblacion", etiqueta="habitantes")

sin_dane = gadm[~gadm["municipio_id"].isin(crosswalk_pob["municipio_id"])]
print(f"\nUnidades de GADM sin código DANE: {len(sin_dane)} de {len(gadm)}")
print(sin_dane[COLS_UNIDAD].head(30).to_string(index=False))

crosswalk_pob.to_csv(CROSSWALK_CSV, index=False)
print("\nGuardado:", CROSSWALK_CSV)

## 7. Cruce con el Sivigila: panel final municipio-semana

Esta sección une los casos de accidente ofídico del Sivigila (evento 100) con el panel climático (sección 5) y con la población (sección 6). Solo requiere la sección 1 y los archivos generados en las secciones 5 y 6. Antes de ejecutarla deben copiarse los archivos `Datos_AAAA_Accidente_ofidico.xlsx` a `proyecto_clima/SIVIGILA/`.

**Decisiones metodológicas**

- **Municipio de ocurrencia** (`COD_DPTO_O` y `COD_MUN_O`), no el de notificación: es donde ocurrió la mordedura y, por tanto, donde se mide el clima.
- **Fecha del caso.** Inicio de síntomas (`INI_SIN`); si precede a la notificación en más de 30 días se considera poco confiable y se usa la fecha de notificación.
- **Semanas.** Cada caso se asigna al mismo bloque de 7 días, contado desde el 1 de enero, que usan CHIRPS y ERA5-Land. No se usa la semana epidemiológica, que empieza en domingo.
- **Exclusiones.** Registros duplicados, casos descartados (`AJUSTE` 6 o D), ocurridos fuera de Colombia, con municipio de ocurrencia desconocido o sin fecha válida.
- **Cruce de códigos.** Los códigos DANE de los casos se traducen a GADM con el puente de la sección 6; no se emparejan nombres del Sivigila.
- **Ceros.** Las combinaciones municipio-semana sin notificación se completan con cero casos.
- **Cobertura.** El año epidemiológico no coincide con el calendario, por lo que algunas semanas quedan cubiertas solo en parte. Para modelar se usa `semana_completa == True`.

### 7.1 Carga y depuración del Sivigila

In [ ]:
def parse_fecha(serie):
    """Fechas del Sivigila ('1/11/2015 12:00:00 a. m.', día/mes/año) a datetime."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.dt.normalize()
    texto = serie.astype(str).str.strip().str.split(" ").str[0]
    fecha = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    pendientes = fecha.isna() & serie.notna()
    if pendientes.any():
        fecha[pendientes] = pd.to_datetime(texto[pendientes], errors="coerce")
    return fecha


COLUMNAS_SIVIGILA = [
    "CONSECUTIVE", "ANO", "FEC_NOT", "INI_SIN", "COD_PAIS_O",
    "COD_DPTO_O", "COD_MUN_O", "Municipio_ocurrencia", "AJUSTE",
]


def cargar_sivigila(carpeta):
    """Lee los Excel del Sivigila, depura y devuelve (casos, bitácora de exclusiones, (año mínimo, año máximo))."""
    archivos = sorted(glob.glob(os.path.join(carpeta, "Datos_*_Accidente_ofidico.xlsx")))
    if not archivos:
        raise FileNotFoundError(f"No hay archivos Datos_*_Accidente_ofidico.xlsx en {carpeta}")

    marcos = []
    for a in archivos:
        df = pd.read_excel(a, usecols=lambda c: c in COLUMNAS_SIVIGILA)
        faltan = set(COLUMNAS_SIVIGILA) - set(df.columns)
        if faltan:
            raise ValueError(f"{os.path.basename(a)}: faltan las columnas {sorted(faltan)}")
        marcos.append(df)
        print(f"  {os.path.basename(a)}: {len(df):,} filas")
    df = pd.concat(marcos, ignore_index=True)

    anios = (int(df["ANO"].min()), int(df["ANO"].max()))
    bitacora = {"Filas leídas": len(df)}

    def excluir(df, mascara, motivo):
        bitacora[motivo] = int(mascara.sum())
        return df[~mascara].copy()

    df = excluir(df, df.duplicated("CONSECUTIVE"), "Excluidos: registros duplicados")

    ajuste = df["AJUSTE"].astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)
    df = excluir(df, ajuste.isin(["6", "D"]), "Excluidos: descartados / error de digitación")

    df = excluir(df, df["COD_PAIS_O"] != 170, "Excluidos: ocurrencia fuera de Colombia")
    df = excluir(df, df["COD_MUN_O"].fillna(0).astype(int) == 0, "Excluidos: municipio de ocurrencia desconocido")

    # Fecha del caso: inicio de síntomas si es confiable; si no, fecha de notificación.
    f_not, f_ini = parse_fecha(df["FEC_NOT"]), parse_fecha(df["INI_SIN"])
    retraso = (f_not - f_ini).dt.days
    usar_ini = retraso.between(0, DIAS_MAX_RETRASO)
    df["fecha_caso"] = f_ini.where(usar_ini, f_not).fillna(f_ini)
    bitacora["(info) fecha tomada de notificación por INI_SIN poco confiable"] = int((~usar_ini).sum())
    df = excluir(df, df["fecha_caso"].isna(), "Excluidos: sin fecha válida")

    df["dpto"] = df["COD_DPTO_O"].astype(int)
    df["dane5"] = (df["dpto"] * 1000 + df["COD_MUN_O"].astype(int)).astype(str).str.zfill(5)
    df = df.rename(columns={"Municipio_ocurrencia": "nombre_sivigila"})

    return df[["CONSECUTIVE", "dane5", "dpto", "nombre_sivigila", "fecha_caso"]], bitacora, anios


def inicio_semana_epi(anio):
    """Domingo de inicio de la semana epidemiológica 1 (la que contiene al menos cuatro días del año)."""
    ene4 = datetime(anio, 1, 4)
    return ene4 - timedelta(days=(ene4.weekday() + 1) % 7)


def asignar_semana(fecha):
    """Inicio del bloque de 7 días, contado desde el 1 de enero (misma rejilla que CHIRPS y ERA5-Land)."""
    ene1 = pd.to_datetime(fecha.dt.year.astype(str) + "-01-01")
    k = (fecha - ene1).dt.days // 7
    return ene1 + pd.to_timedelta(k * 7, unit="D")


print("Leyendo el Sivigila...")
df_siv, bitacora, (ANIO_MIN, ANIO_MAX) = cargar_sivigila(DIR_SIVIGILA)
print(f"\n{len(df_siv):,} casos válidos | años de los archivos: {ANIO_MIN}-{ANIO_MAX}")

### 7.2 Casos con y sin pareja en GADM

In [ ]:
df_clima = pd.read_csv(CLIMA_CSV, parse_dates=["week_start"])
crosswalk = pd.read_csv(CROSSWALK_CSV, dtype={"dane5": str})
pob_df = pd.read_csv(POB_CSV, dtype={"dane5": str})

# Se añaden al puente los códigos manuales del Sivigila; los que apuntan a un id inexistente se ignoran.
ids_panel = set(df_clima["municipio_id"])
manual_casos = {}
for codigo, mid in MANUAL_CASOS.items():
    if mid in ids_panel:
        manual_casos[codigo] = mid
    else:
        print(f"Aviso: MANUAL_CASOS {codigo} apunta a '{mid}', que no existe en el panel; se ignora.")
if manual_casos:
    extra = pd.DataFrame({"dane5": list(manual_casos), "municipio_id": list(manual_casos.values()),
                          "metodo": "manual (casos)"})
    crosswalk = pd.concat([crosswalk[~crosswalk["dane5"].isin(extra["dane5"])], extra], ignore_index=True)

unidades = (
    df_siv.groupby("dane5")
    .agg(nombre=("nombre_sivigila", lambda s: s.mode().iat[0]), casos=("CONSECUTIVE", "size"))
    .reset_index()
)
sin_match = unidades[~unidades["dane5"].isin(crosswalk["dane5"])].copy()
sin_match["motivo"] = np.where(
    sin_match["dane5"].isin(pob_df["dane5"]),
    "municipio del DANE sin pareja en GADM",
    "código ausente en la lista del DANE",
)

total_casos = unidades["casos"].sum()
print(f"Unidades del Sivigila con casos: {len(unidades)} | con pareja en GADM: {len(unidades) - len(sin_match)}")
print(f"Sin correspondencia: {len(sin_match)} unidades, {sin_match['casos'].sum():,} casos "
      f"({sin_match['casos'].sum() / total_casos:.2%} del total)")
if len(sin_match):
    print("\nCasos sin correspondencia, de mayor a menor:")
    print(sin_match.sort_values("casos", ascending=False)
          [["dane5", "nombre", "casos", "motivo"]].head(30).to_string(index=False))
    print("\nUn municipio importante en esta lista se corrige en MANUAL_MATCH (si su código está en la lista "
          "del DANE) o en MANUAL_CASOS (si no lo está), ambos en la sección 1.")

### 7.3 Panel final: casos, población y clima

In [ ]:
# Casos -> municipio de GADM -> semana (rejilla de CHIRPS y ERA5-Land)
casos_geo = df_siv.merge(crosswalk[["dane5", "municipio_id"]], on="dane5", how="left")

sin_cw = casos_geo["municipio_id"].isna()
bitacora["Excluidos: sin correspondencia con GADM"] = int(sin_cw.sum())
casos_geo = casos_geo[~sin_cw].copy()

casos_geo["week_start"] = asignar_semana(casos_geo["fecha_caso"])
fuera_panel = ~casos_geo["week_start"].isin(df_clima["week_start"].unique())
bitacora["Excluidos: fecha fuera del panel climático"] = int(fuera_panel.sum())
casos_geo = casos_geo[~fuera_panel]

conteo = casos_geo.groupby(["municipio_id", "week_start"]).size().rename("casos").reset_index()

# Panel completo: cada municipio en cada semana, con cero casos donde no hubo notificación.
panel = df_clima.merge(conteo, on=["municipio_id", "week_start"], how="left")
panel["casos"] = panel["casos"].fillna(0).astype(int)

# Un solo código DANE por municipio_id: dos códigos pueden apuntar al mismo polígono
# (Belén de Bajirá), y se prefiere el que tiene población en la lista del DANE.
codigo_por_unidad = (
    crosswalk.assign(en_pob=crosswalk["dane5"].isin(pob_df["dane5"]))
    .sort_values(["municipio_id", "en_pob", "dane5"], ascending=[True, False, True])
    .drop_duplicates("municipio_id")[["municipio_id", "dane5"]]
    .rename(columns={"dane5": "dane_code"})
)
n_filas = len(panel)
panel = panel.merge(codigo_por_unidad, on="municipio_id", how="left")
assert len(panel) == n_filas, "El código DANE duplicó filas del panel"

# Calendario
ws = panel["week_start"]
ene1 = pd.to_datetime(ws.dt.year.astype(str) + "-01-01")
panel["anio"] = ws.dt.year
panel["semana_anio"] = (ws - ene1).dt.days // 7 + 1   # de 1 a 53

# Población del DANE: valor anual que se repite en todas las semanas del año.
pob_anual = pob_df[["dane5", "anio", "total", "rural"]].rename(
    columns={"dane5": "dane_code", "total": "poblacion_total", "rural": "poblacion_rural"}
).drop_duplicates(["dane_code", "anio"])
panel = panel.merge(pob_anual, on=["dane_code", "anio"], how="left")

assert len(panel) == len(df_clima), "El panel final tiene más filas que el panel climático"
assert panel["casos"].sum() == len(casos_geo), "Los casos del panel no coinciden con los casos asignados"
assert not panel.duplicated(["municipio_id", "week_start"]).any(), "Hay pares municipio-semana duplicados"

# Días del bloque que caen en el mismo año calendario (el bloque 53 tiene solo uno o dos).
fin_anio = pd.to_datetime(ws.dt.year.astype(str) + "-12-31")
fin_bloque = pd.concat([ws + pd.Timedelta(days=6), fin_anio], axis=1).min(axis=1)
panel["dias_semana"] = (fin_bloque - ws).dt.days + 1

# Días del bloque cubiertos por los archivos del Sivigila cargados: el año epidemiológico
# no coincide con el calendario (el de 2015 empieza el 4 de enero).
COB_INI = pd.Timestamp(inicio_semana_epi(ANIO_MIN))
COB_FIN = pd.Timestamp(inicio_semana_epi(ANIO_MAX + 1) - timedelta(days=1))
ini_cub = ws.where(ws >= COB_INI, COB_INI)
fin_cub = fin_bloque.where(fin_bloque <= COB_FIN, COB_FIN)
panel["dias_cubiertos"] = ((fin_cub - ini_cub).dt.days + 1).clip(lower=0)
panel["semana_completa"] = panel["dias_cubiertos"] == 7

columnas = [
    "municipio_id", "dane_code", "departamento", "municipio", "tipo_administrativo",
    "week_start", "anio", "semana_anio", "dias_semana", "dias_cubiertos", "semana_completa",
    "casos", "poblacion_total", "poblacion_rural",
    "precip_mm", "temp_mean", "temp_max", "temp_min", "rh_mean",
]
panel = panel[columnas].sort_values(["municipio_id", "week_start"]).reset_index(drop=True)

# Conciliación de casos y resumen del panel
print("Conciliación de casos")
for motivo, n in bitacora.items():
    print(f"  {motivo:<62}{n:>9,}")
print(f"  {'Casos en el panel':<62}{int(panel['casos'].sum()):>9,}")

completas = panel[panel["semana_completa"]]
print(f"\nPanel: {len(panel):,} filas | {panel['municipio_id'].nunique()} unidades | "
      f"{panel['week_start'].nunique()} semanas ({panel['week_start'].min():%Y-%m-%d} a {panel['week_start'].max():%Y-%m-%d})")
print(f"Semanas completas: {completas['week_start'].nunique()} de {panel['week_start'].nunique()}")
print(f"Celdas municipio-semana con 0 casos (semanas completas): {(completas['casos'] == 0).mean():.1%}")
print(f"Unidades con al menos un caso: {panel.groupby('municipio_id')['casos'].sum().gt(0).sum()}")

anual = panel.groupby("anio")["casos"].sum()
unico = panel.drop_duplicates(["municipio_id", "anio"]).groupby("anio")
print("\nCasos por año:")
print(anual.to_string())
print("\nIncidencia nacional anual (por 100.000 habitantes):")
print(pd.DataFrame({
    "casos": anual,
    "inc_total": (anual / unico["poblacion_total"].sum() * 1e5).round(1),
    "inc_rural": (anual / unico["poblacion_rural"].sum() * 1e5).round(1),
}).to_string())

print(f"\nFilas sin población: {panel['poblacion_total'].isna().mean():.2%}")
falta_pob = panel[panel["poblacion_total"].isna()].groupby(["municipio_id", "municipio"])["anio"].agg(["min", "max", "nunique"])
if len(falta_pob):
    print("Unidades con población faltante (años afectados):")
    print(falta_pob.rename(columns={"min": "desde", "max": "hasta", "nunique": "n_años"}).to_string())
print("\nFaltantes en variables climáticas:")
print(panel[["precip_mm", "temp_mean", "temp_max", "temp_min", "rh_mean"]].isna().sum().to_string())

panel.to_csv(PANEL_CSV, index=False)
print("\nGuardado:", PANEL_CSV)

## 8. Conjunto de modelado: rezagos, acumulados y anomalías

Esta sección convierte el panel de la sección 7 en `dataset_modelado.csv`: una fila por municipio y semana, con la variable objetivo (`casos`), la exposición (población) y variables explicativas construidas únicamente con información anterior o contemporánea a la semana modelada.

### Fundamento de las decisiones

| Decisión | Fundamento |
|---|---|
| Acumulados de lluvia de 4, 8, 13 y 26 semanas (aproximadamente 1, 2, 3 y 6 meses) | Misma lógica multiescala del *Standardized Precipitation Index* (SPI), recomendado por la Organización Meteorológica Mundial (OMM, 2009; guía WMO-No. 1090, 2012), que se calcula a 1, 3, 6 y 12 meses porque cada escala captura procesos distintos. |
| Anomalías (puntaje z) en lugar de valores crudos | Un mismo valor de lluvia o de humedad significa cosas distintas según el municipio y la época del año. En Colombia la lluvia solo explica la incidencia en regiones con estación seca marcada (Bravo-Vega et al., 2022), y en Sri Lanka una humedad menor que la esperada para la temporada se asoció con más mordeduras (Ediriweera et al., 2018). |
| Humedad relativa como variable principal | Los cambios de humedad aumentaron el riesgo en los cuatro municipios de Rondônia, mientras que la lluvia y la temperatura superficial no lo hicieron (Ferreira et al., 2020). |
| Rezagos de 1, 2, 4 y 8 semanas | Chaves et al. (2015), en Costa Rica, hallaron asociaciones con rezagos de meses (temperatura con un mes de rezago; lluvia con rezagos largos) y relación con El Niño–Oscilación del Sur a escala interanual. Los acumulados de 13 y 26 semanas cubren esos horizontes largos. Esos estudios son mensuales, por lo que trasladarlos a semanas es una aproximación. |
| Casos de semanas anteriores | Los casos presentan autocorrelación significativa (Chaves et al., 2015). Se incluyen rezagos municipales y la suma departamental de las cuatro semanas previas, que es menos dispersa. |
| Población como exposición | En los modelos de conteo (Poisson y binomial negativa) se incluye el logaritmo de la población como *offset*, de modo que el modelo explique la tasa y no el tamaño del municipio. Se ofrecen la población total y la rural. |
| `sin_semana` y `cos_semana` | Términos de Fourier de primer orden: forma habitual de representar la estacionalidad anual sin tratar las semanas 52 y 1 como lejanas. |
| Partición temporal | En series de tiempo no se debe mezclar al azar: el entrenamiento comprende los años hasta `ANIO_FIN_ENTRENAMIENTO` y la prueba los posteriores. |

### Cuidados metodológicos

- **Sin fuga de información.** La climatología (media y desviación por municipio y semana del año) se calcula solo con los años de entrenamiento y luego se aplica a todos. Se comprobó que alterar el clima del periodo de prueba no cambia ninguna variable de entrenamiento.
- **Climatología corta.** La OMM sugiere al menos 30 años para el SPI; aquí la base tiene siete. Para estabilizarla se agrupan las semanas vecinas (±2) y se usa el puntaje z de log(1 + acumulado) como aproximación al SPI, no el ajuste gamma completo. Esta decisión debe declararse como limitación.
- **Rezagos y momento de uso.** Las variables con sufijo `_lagK` y las `*_prev` solo usan semanas hasta t−K. Para predecir con horizonte de H semanas, el entrenamiento debe usar únicamente variables con rezago de al menos H.
- **Filas excluidas.** Semanas con cobertura parcial del Sivigila, bloque 53 y unidades sin datos climáticos.
- **Primeras semanas de 2015.** No tienen historia suficiente para los acumulados largos y quedan con `features_completas = False`; no se eliminan para que la decisión se tome en el modelado.

### 8.1 Variables derivadas

In [ ]:
# Parámetros (justificados en la tabla anterior)
LAGS = [1, 2, 4, 8]    # rezagos directos, en semanas
ACUM = [4, 8, 13, 26]  # acumulados de lluvia, aproximadamente 1, 2, 3 y 6 meses (como SPI-1, -2, -3 y -6)
VENTANA_CLIM = 2       # la climatología agrupa las semanas vecinas (±2) para estabilizarse
CLIP_Z = 5             # las anomalías se limitan a ±5 desviaciones
SD_MIN = {"precip": 0.1, "rh": 1.0, "temp": 0.1}   # desviación mínima, para evitar z enormes en zonas casi constantes

# Serie regular: se descarta el bloque 53, que tiene solo uno o dos días del año y se solapa con el siguiente.
panel = pd.read_csv(PANEL_CSV, dtype={"dane_code": str}, parse_dates=["week_start"])
serie = panel[panel["semana_anio"] <= 52].copy()
serie["casos_valido"] = serie["casos"].where(serie["semana_completa"])   # NaN donde la cobertura es parcial

ids = sorted(serie["municipio_id"].unique())
semanas = np.sort(serie["week_start"].unique())
anios_semana = pd.DatetimeIndex(semanas).year.values
anios = np.unique(anios_semana)

assert serie.groupby("municipio_id").size().nunique() == 1, "Panel desbalanceado"
assert all((anios_semana == a).sum() == 52 for a in anios), "Cada año debe tener 52 semanas"

es_base = anios <= ANIO_FIN_ENTRENAMIENTO
print(f"Municipios: {len(ids)} | semanas: {len(semanas)} | años: {anios.min()}-{anios.max()}")
print(f"Climatología base (entrenamiento): {anios[es_base].min()}-{anios[es_base].max()}")


def ancho(col):
    """Matriz de semanas por municipios."""
    return (serie.pivot(index="week_start", columns="municipio_id", values=col)
            .reindex(columns=ids).sort_index())


def anomalia(matriz, sd_min):
    """Puntaje z frente a la climatología (media y desviación por municipio y semana del año, solo años base)."""
    x = matriz.to_numpy(dtype=float).reshape(len(anios), 52, len(ids))   # año, semana, municipio
    grupo = np.stack([np.roll(x, s, axis=1) for s in range(-VENTANA_CLIM, VENTANA_CLIM + 1)])
    base = grupo[:, es_base]   # solo años de entrenamiento: evita la fuga de información
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mu = np.nanmean(base, axis=(0, 1))
        sd = np.nanstd(base, axis=(0, 1), ddof=1)
    z = np.clip((x - mu) / np.maximum(sd, sd_min), -CLIP_Z, CLIP_Z)
    return pd.DataFrame(z.reshape(len(semanas), len(ids)), index=matriz.index, columns=ids)


# Variables climáticas: rezagos, acumulados y anomalías
precip, rh, temp = ancho("precip_mm"), ancho("rh_mean"), ancho("temp_mean")
F = {}

for k in LAGS:
    F[f"precip_mm_lag{k}"] = precip.shift(k)
    F[f"rh_mean_lag{k}"] = rh.shift(k)
    F[f"temp_mean_lag{k}"] = temp.shift(k)

for k in ACUM:
    F[f"precip_sum{k}w"] = precip.rolling(k, min_periods=k).sum()
F["rh_mean4w"] = rh.rolling(4, min_periods=4).mean()
F["temp_mean4w"] = temp.rolling(4, min_periods=4).mean()

for k in ACUM:   # lluvia: z de log(1 + acumulado), porque la lluvia es asimétrica
    F[f"z_precip_{k}w"] = anomalia(np.log1p(F[f"precip_sum{k}w"]), SD_MIN["precip"])
F["z_rh_1w"] = anomalia(rh, SD_MIN["rh"])
F["z_rh_4w"] = anomalia(F["rh_mean4w"], SD_MIN["rh"])
F["z_temp_1w"] = anomalia(temp, SD_MIN["temp"])
F["z_temp_4w"] = anomalia(F["temp_mean4w"], SD_MIN["temp"])

# Variables autorregresivas: casos de semanas anteriores, nunca los de la semana que se predice.
casos_w = ancho("casos_valido")
F["casos_lag1"] = casos_w.shift(1)
F["casos_lag2"] = casos_w.shift(2)
F["casos_lag4"] = casos_w.shift(4)
F["casos_sum4w_prev"] = casos_w.rolling(4, min_periods=4).sum().shift(1)
F["casos_sum13w_prev"] = casos_w.rolling(13, min_periods=13).sum().shift(1)

dpto_de = serie.drop_duplicates("municipio_id").set_index("municipio_id")["departamento"].reindex(ids)
casos_dpto = casos_w.T.groupby(dpto_de.values).sum(min_count=1).T
dpto_4w = casos_dpto.rolling(4, min_periods=4).sum().shift(1)[dpto_de.values]
dpto_4w.columns = ids
F["casos_dpto_sum4w_prev"] = dpto_4w

print(f"{len(F)} variables derivadas calculadas")

### 8.2 Ensamblado, filtros, partición temporal y guardado

In [ ]:
# Tabla larga (municipio-semana)
idx = pd.MultiIndex.from_product([sorted(serie["week_start"].unique()), ids],
                                 names=["week_start", "municipio_id"])
derivadas = pd.DataFrame({n: df.to_numpy().ravel() for n, df in F.items()}, index=idx).reset_index()

datos = serie.drop(columns="casos_valido").merge(derivadas, on=["week_start", "municipio_id"], how="left")

# Exposición: logaritmo de la población, para usarlo como offset en los modelos de conteo.
for c in ("poblacion_total", "poblacion_rural"):
    datos[f"log_{c}"] = np.log(datos[c].where(datos[c] > 0))

# Estacionalidad (Fourier de primer orden) y partición temporal
datos["sin_semana"] = np.sin(2 * np.pi * datos["semana_anio"] / 52)
datos["cos_semana"] = np.cos(2 * np.pi * datos["semana_anio"] / 52)
datos["particion"] = np.where(datos["anio"] <= ANIO_FIN_ENTRENAMIENTO, "entrenamiento", "prueba")

# Filas válidas: semana con cobertura completa y municipio con clima
n_inicial, casos_inicial = len(datos), int(datos["casos"].sum())

incompleta = ~datos["semana_completa"]
sin_clima = datos["precip_mm"].isna() | datos["temp_mean"].isna() | datos["rh_mean"].isna()

print(f"Filas del panel sin el bloque 53: {n_inicial:,}")
print(f"  Excluidas por semana con cobertura parcial: {int(incompleta.sum()):,}")
print(f"  Excluidas por municipio sin clima:          {int((sin_clima & ~incompleta).sum()):,}")

datos = datos[~incompleta & ~sin_clima].copy()

CLIMA = ["precip_mm", "temp_mean", "temp_max", "temp_min", "rh_mean"]
LAGS_COLS = [f"{v}_lag{k}" for k in LAGS for v in ("precip_mm", "rh_mean", "temp_mean")]
ACUM_COLS = [f"precip_sum{k}w" for k in ACUM] + ["rh_mean4w", "temp_mean4w"]
ANOM_COLS = [f"z_precip_{k}w" for k in ACUM] + ["z_rh_1w", "z_rh_4w", "z_temp_1w", "z_temp_4w"]
AUTO_COLS = ["casos_lag1", "casos_lag2", "casos_lag4", "casos_sum4w_prev",
             "casos_sum13w_prev", "casos_dpto_sum4w_prev"]
CAL_COLS = ["sin_semana", "cos_semana"]
EXPO_COLS = ["poblacion_total", "poblacion_rural", "log_poblacion_total", "log_poblacion_rural"]
FEATURES = CLIMA + LAGS_COLS + ACUM_COLS + ANOM_COLS + AUTO_COLS + CAL_COLS

# Las primeras semanas de 2015 no tienen historia suficiente para algunos rezagos y acumulados.
datos["features_completas"] = datos[FEATURES].notna().all(axis=1)

assert datos["casos"].notna().all()

ID_COLS = ["municipio_id", "dane_code", "departamento", "municipio", "tipo_administrativo",
           "week_start", "anio", "semana_anio", "particion", "features_completas"]
dataset = (datos[ID_COLS + ["casos"] + EXPO_COLS + FEATURES]
           .sort_values(["municipio_id", "week_start"]).reset_index(drop=True))

print(f"\ndataset_modelado: {len(dataset):,} filas x {dataset.shape[1]} columnas")
print(f"   Casos: {int(dataset['casos'].sum()):,} de {casos_inicial:,} del panel "
      f"(el resto está en semanas parciales, en el bloque 53 o en municipios sin clima)")
print(f"   Filas con todas las variables completas: {dataset['features_completas'].mean():.1%}")
print(f"   Filas sin población: {dataset['poblacion_total'].isna().sum():,}")

resumen = dataset.groupby("particion").agg(
    filas=("casos", "size"), casos=("casos", "sum"),
    pct_ceros=("casos", lambda s: (s == 0).mean()),
    primer_anio=("anio", "min"), ultimo_anio=("anio", "max"),
)
print("\nPartición temporal:")
print(resumen.to_string())

print(f"\nVariables: {len(CLIMA)} clima | {len(LAGS_COLS)} rezagos | {len(ACUM_COLS)} acumulados | "
      f"{len(ANOM_COLS)} anomalías | {len(AUTO_COLS)} autorregresivas | {len(CAL_COLS)} estacionalidad")

dataset.to_csv(MODELADO_CSV, index=False)
print("\nGuardado:", MODELADO_CSV)

### Diccionario de variables de `dataset_modelado.csv`

| Grupo | Columnas |
|---|---|
| Identificación | `municipio_id`, `dane_code`, `departamento`, `municipio`, `tipo_administrativo`, `week_start`, `anio`, `semana_anio` |
| Control | `particion` (entrenamiento o prueba), `features_completas` |
| Objetivo | `casos` |
| Exposición | `poblacion_total`, `poblacion_rural`, `log_poblacion_total`, `log_poblacion_rural` |
| Clima de la semana | `precip_mm`, `temp_mean`, `temp_max`, `temp_min`, `rh_mean` |
| Rezagos (1, 2, 4 y 8 semanas) | `precip_mm_lagK`, `rh_mean_lagK`, `temp_mean_lagK` |
| Acumulados y medias móviles | `precip_sum4w`, `precip_sum8w`, `precip_sum13w`, `precip_sum26w`, `rh_mean4w`, `temp_mean4w` |
| Anomalías (puntaje z) | `z_precip_4w`, `z_precip_8w`, `z_precip_13w`, `z_precip_26w`, `z_rh_1w`, `z_rh_4w`, `z_temp_1w`, `z_temp_4w` |
| Casos previos | `casos_lag1`, `casos_lag2`, `casos_lag4`, `casos_sum4w_prev`, `casos_sum13w_prev`, `casos_dpto_sum4w_prev` |
| Estacionalidad | `sin_semana`, `cos_semana` |

## 9. Capa espacial para QGIS

Esta sección resume el panel por municipio y por departamento, y lo une a los polígonos de GADM en un GeoPackage (`ofidico_municipal.gpkg`) que se abre directamente en QGIS. La tabla semanal, de unas 593.000 filas, no sirve para cartografiar; aquí se condensa en una fila por unidad.

**Contenido de la capa `municipios`**

| Grupo | Columnas |
|---|---|
| Identificación | `municipio_id`, `dane_code`, `departamento`, `municipio`, `tipo_administrativo` |
| Casos | `casos_AAAA` (por año), `casos_total`, `casos_entren` (hasta `ANIO_FIN_ENTRENAMIENTO`), `casos_prueba` (después) |
| Incidencia (por 100.000 habitantes) | `inc_AAAA` (anual), `inc_media` (promedio anual 2015–2024), `inc_media_rural` |
| Tasa suavizada | `inc_eb`: tasa por Bayes empírico, que se usa para los mapas y los puntos calientes |
| Población | `poblacion_media`, `poblacion_rural_media` |
| Clima medio | `precip_anual_mm`, `temp_media_c`, `rh_media_pct` |
| Estadística espacial | `lisa_cluster`, `lisa_i`, `lisa_p`, `gi_z`, `gi_p`, `gi_clase`, `gi_clase_fdr` |

La capa `departamentos` trae los casos y la incidencia por año y la incidencia media.

**Decisiones metodológicas**

- **Incidencia = casos / persona-años.** Los casos y la población se cuentan solo en los años con población disponible, de modo que numerador y denominador sean coherentes (Belén de Bajirá, por ejemplo, no tiene población en 2015–2017).
- **Suavizado por Bayes empírico (Marshall, 1991).** Un municipio de 5.000 habitantes con un solo caso tendría una tasa de 20 por 100.000 al año por puro azar. El suavizado acerca las tasas de los municipios pequeños a la tasa nacional, en proporción a su inestabilidad; sin él, los mapas de puntos calientes quedan dominados por municipios diminutos.
- **Vecindad tipo reina.** Dos municipios son vecinos si comparten un borde o un vértice. Las islas (San Andrés y Providencia) no tienen vecinos y no aportan al análisis.
- **Estadísticos.** El índice de Moran global indica si hay agrupamiento espacial en general; el Moran local (LISA) localiza los grupos alto-alto y bajo-bajo y los atípicos alto-bajo y bajo-alto; el Getis-Ord Gi\* identifica puntos calientes y fríos. Los tres usan 999 permutaciones con semilla fija, por lo que son reproducibles.
- **Corrección por pruebas múltiples.** Con unas 1.100 pruebas simultáneas se esperan alrededor de 55 falsos positivos al nivel del 5 %. Por eso se entrega también `gi_clase_fdr`, con control de la tasa de falsos descubrimientos (Benjamini y Hochberg, 1995); es la versión que se recomienda informar.
- **Año calendario aproximado.** Los casos se asignan a bloques de 7 días desde el 1 de enero; los primeros días de 2015 y los últimos de 2024 quedan fuera de la cobertura del Sivigila, de modo que esos dos años están ligeramente subestimados.

**Apertura en QGIS.** *Capa, Añadir capa, Añadir capa vectorial*, seleccionar `ofidico_municipal.gpkg` y marcar las dos capas. Simbología sugerida: graduada sobre `inc_eb` (Jenks o cuantiles) para el mapa de riesgo; categorizada sobre `lisa_cluster` o `gi_clase_fdr` para los puntos calientes; graduada sobre `precip_anual_mm` o `temp_media_c` para el clima. Para el análisis semanal con el Controlador Temporal, se une `panel_ofidico_clima_semanal.csv` a la capa por `municipio_id` (como texto) y se filtra a pocas semanas o a un departamento, porque la unión con todas las filas es lenta.

### 9.1 Tablas por municipio y departamento

In [ ]:
# Población con la que se calcula la tasa suavizada y los puntos calientes: "total" o "rural".
POB_HOTSPOT = "total"

panel = pd.read_csv(PANEL_CSV, dtype={"dane_code": str}, parse_dates=["week_start"])
assert not panel.duplicated(["municipio_id", "week_start"]).any(), "Panel con pares municipio-semana duplicados"
anios = sorted(panel["anio"].unique())

# Municipio y año: casos y población (las poblaciones menores o iguales a cero se tratan como faltantes).
anual = (
    panel.groupby(["municipio_id", "anio"])
    .agg(casos=("casos", "sum"), pob_total=("poblacion_total", "first"), pob_rural=("poblacion_rural", "first"))
    .reset_index()
)
for c in ("pob_total", "pob_rural"):
    anual[c] = anual[c].where(anual[c] > 0)

casos_anio = anual.pivot(index="municipio_id", columns="anio", values="casos")
pob_anio = anual.pivot(index="municipio_id", columns="anio", values="pob_total")

tabla = pd.DataFrame(index=casos_anio.index)
for a in anios:
    tabla[f"casos_{a}"] = casos_anio[a]
for a in anios:
    tabla[f"inc_{a}"] = casos_anio[a] / pob_anio[a] * 1e5   # casos por 100.000 habitantes en el año

por_muni = anual.groupby("municipio_id")
tabla["casos_total"] = por_muni["casos"].sum()
tabla["casos_entren"] = anual[anual["anio"] <= ANIO_FIN_ENTRENAMIENTO].groupby("municipio_id")["casos"].sum()
tabla["casos_prueba"] = anual[anual["anio"] > ANIO_FIN_ENTRENAMIENTO].groupby("municipio_id")["casos"].sum()
tabla[["casos_entren", "casos_prueba"]] = tabla[["casos_entren", "casos_prueba"]].fillna(0)
tabla["poblacion_media"] = por_muni["pob_total"].mean()
tabla["poblacion_rural_media"] = por_muni["pob_rural"].mean()


def tasa_periodo(pob_col):
    """Casos y persona-años solo en los años con población disponible (numerador y denominador coherentes)."""
    d = anual.dropna(subset=[pob_col])
    return d.groupby("municipio_id").agg(casos=("casos", "sum"), persona_anios=(pob_col, "sum"))


tp_total, tp_rural = tasa_periodo("pob_total"), tasa_periodo("pob_rural")
tabla["inc_media"] = tp_total["casos"] / tp_total["persona_anios"] * 1e5   # por 100.000 habitantes al año
tabla["inc_media_rural"] = tp_rural["casos"] / tp_rural["persona_anios"] * 1e5


def suavizar_eb(casos, persona_anios):
    """Bayes empírico global (Marshall, 1991): acerca las tasas de los municipios pequeños a la tasa nacional."""
    r = casos / persona_anios
    m = casos.sum() / persona_anios.sum()
    s2 = max((persona_anios * (r - m) ** 2).sum() / persona_anios.sum() - m / persona_anios.mean(), 0)
    w = s2 / (s2 + m / persona_anios)
    return w * r + (1 - w) * m


base = tp_total if POB_HOTSPOT == "total" else tp_rural
base = base[base["persona_anios"] > 0]
tabla["inc_eb"] = suavizar_eb(base["casos"], base["persona_anios"]) * 1e5

# Clima medio de cada municipio (normales del periodo)
clima_medio = (
    panel[panel["semana_anio"] <= 52]
    .groupby(["municipio_id", "anio"])
    .agg(p=("precip_mm", lambda x: x.sum(min_count=1)), t=("temp_mean", "mean"), h=("rh_mean", "mean"))
    .groupby("municipio_id").mean()
)
tabla["precip_anual_mm"] = clima_medio["p"]
tabla["temp_media_c"] = clima_medio["t"]
tabla["rh_media_pct"] = clima_medio["h"]

# Departamentos: casos y población sumados de sus municipios
dpto = panel.drop_duplicates("municipio_id").set_index("municipio_id")["departamento"]
anual_dep = anual.dropna(subset=["pob_total"]).assign(departamento=lambda d: d["municipio_id"].map(dpto))
dep_anio = anual_dep.groupby(["departamento", "anio"]).agg(casos=("casos", "sum"), pob=("pob_total", "sum")).reset_index()

tabla_dep = pd.DataFrame(index=sorted(dpto.unique()))
tabla_dep.index.name = "departamento"
for a in anios:
    s = dep_anio[dep_anio["anio"] == a].set_index("departamento")
    tabla_dep[f"casos_{a}"] = s["casos"]
    tabla_dep[f"inc_{a}"] = s["casos"] / s["pob"] * 1e5
tot = dep_anio.groupby("departamento").agg(casos=("casos", "sum"), persona_anios=("pob", "sum"))
tabla_dep["casos_total"] = tot["casos"]
tabla_dep["inc_media"] = tot["casos"] / tot["persona_anios"] * 1e5

# Controles
print(f"Municipios: {len(tabla)} | departamentos: {len(tabla_dep)} | años: {anios[0]}-{anios[-1]}")
print(f"Casos en la tabla municipal: {int(tabla['casos_total'].sum()):,} (deben coincidir con 'Casos en el panel' de la sección 7.3)")
print(f"Incidencia media nacional: {tp_total['casos'].sum() / tp_total['persona_anios'].sum() * 1e5:.1f} por 100.000 habitantes al año")
print(f"Municipios sin tasa (sin población): {tabla['inc_media'].isna().sum()}")
eb_global = (tabla["inc_eb"].reindex(base.index) * base["persona_anios"]).sum() / base["persona_anios"].sum()
print(f"Tasa global tras el suavizado: {eb_global:.1f} (debe ser cercana a la cruda; una diferencia grande indicaría un problema)")
print("\nDiez departamentos con mayor incidencia media (por 100.000 habitantes al año):")
print(tabla_dep["inc_media"].sort_values(ascending=False).head(10).round(1).to_string())
print("\nDiez municipios con mayor tasa suavizada (Bayes empírico):")
top = tabla.join(panel.drop_duplicates("municipio_id").set_index("municipio_id")[["departamento", "municipio"]])
print(top.sort_values("inc_eb", ascending=False)[["departamento", "municipio", "casos_total", "inc_media", "inc_eb"]]
      .head(10).round(1).to_string())

# Plausibilidad del clima medio
cols_clima = ["precip_anual_mm", "temp_media_c", "rh_media_pct"]
print("\nClima medio por municipio:")
print(tabla[cols_clima].describe().round(1))
print("Valores <= 0:", (tabla[cols_clima] <= 0).sum().to_dict(), "| vacíos:", tabla[cols_clima].isna().sum().to_dict())
if "dataset" in globals():   # solo si la sección 8 se ejecutó en esta sesión
    print("\nClima en el conjunto de modelado:")
    print(dataset[["precip_mm", "temp_mean", "rh_mean"]].describe().round(1))
    print("temp_mean <= 0:", (dataset["temp_mean"] <= 0).sum())

### 9.2 Polígonos y capas

In [ ]:
# Polígonos de GADM con los atributos de las tablas anteriores
geo = cargar_gadm()[["municipio_id", "geometry"]]
meta_mun = panel[["municipio_id", "dane_code", "departamento", "municipio", "tipo_administrativo"]].drop_duplicates("municipio_id")

mun = geo.merge(meta_mun, on="municipio_id", how="inner").merge(tabla.reset_index(), on="municipio_id", how="left")
assert len(mun) == len(tabla), f"Polígonos ({len(mun)}) y tabla ({len(tabla)}) no coinciden"
assert not mun.geometry.is_empty.any(), "Hay polígonos vacíos"

dep = (
    mun[["departamento", "geometry"]].dissolve(by="departamento").reset_index()
    .merge(tabla_dep.reset_index(), on="departamento", how="left")
)
print(f"Capas en memoria: municipios ({len(mun)}) y departamentos ({len(dep)}) | CRS: {mun.crs}")

### 9.3 Autocorrelación espacial: Moran, LISA y Getis-Ord Gi\*

Los tres estadísticos se calculan sobre la tasa suavizada por Bayes empírico del periodo completo.

In [ ]:
def umbral_fdr(p, alfa=0.05):
    """Benjamini-Hochberg: mayor valor p que puede declararse significativo controlando la FDR (-1 si ninguno)."""
    p = np.sort(np.asarray(p, dtype=float))
    m = len(p)
    ok = np.nonzero(p <= alfa * np.arange(1, m + 1) / m)[0]
    return p[ok.max()] if len(ok) else -1.0


TASA = "inc_eb"
SEMILLA = 12345
PERMUTACIONES = 999
ALFA = 0.05

sub = mun[np.isfinite(mun[TASA])].reset_index(drop=True)
print(f"Unidades analizadas: {len(sub)} de {len(mun)} (se excluyen las que no tienen tasa)")

w = weights.Queen.from_dataframe(sub, use_index=False, silence_warnings=True)   # vecinos: comparten borde o vértice
print(f"Unidades sin vecinos (islas): {len(w.islands)}")
y = sub[TASA].to_numpy()

# Moran global
np.random.seed(SEMILLA)
w.transform = "r"
moran = Moran(y, w, permutations=PERMUTACIONES)
print(f"\nI de Moran global = {moran.I:.3f} (esperado bajo aleatoriedad = {moran.EI:.4f}, p simulado = {moran.p_sim:.3f})")

# LISA (Moran local)
lisa = Moran_Local(y, w, permutations=PERMUTACIONES, seed=SEMILLA)
nombres_q = {1: "Alto-Alto", 2: "Bajo-Alto", 3: "Bajo-Bajo", 4: "Alto-Bajo"}
sub["lisa_i"] = lisa.Is
sub["lisa_p"] = lisa.p_sim
sub["lisa_cluster"] = np.where(lisa.p_sim < ALFA, pd.Series(lisa.q).map(nombres_q), "No significativo")

# Getis-Ord Gi* (puntos calientes y fríos), con pesos binarios
wb = weights.Queen.from_dataframe(sub, use_index=False, silence_warnings=True)
wb.transform = "B"
gi = G_Local(y, wb, transform="B", star=True, permutations=PERMUTACIONES, seed=SEMILLA)
sub["gi_z"] = gi.Zs
sub["gi_p"] = gi.p_sim
umbral = umbral_fdr(gi.p_sim, ALFA)   # control de falsos descubrimientos: hay muchas pruebas simultáneas
sub["gi_clase"] = np.select(
    [(gi.p_sim < ALFA) & (gi.Zs > 0), (gi.p_sim < ALFA) & (gi.Zs < 0)], ["Hotspot", "Coldspot"], "No significativo")
sub["gi_clase_fdr"] = np.select(
    [(gi.p_sim <= umbral) & (gi.Zs > 0), (gi.p_sim <= umbral) & (gi.Zs < 0)], ["Hotspot", "Coldspot"], "No significativo")

cols_nuevas = ["lisa_i", "lisa_p", "lisa_cluster", "gi_z", "gi_p", "gi_clase", "gi_clase_fdr"]
mun = mun.drop(columns=[c for c in cols_nuevas if c in mun.columns]).merge(
    sub[["municipio_id"] + cols_nuevas], on="municipio_id", how="left")
for c in ("lisa_cluster", "gi_clase", "gi_clase_fdr"):
    mun[c] = mun[c].fillna("Sin dato")

print(f"\nGrupos LISA (p < {ALFA}):")
print(mun["lisa_cluster"].value_counts().to_string())
print("\nGetis-Ord Gi*, sin corregir y con FDR:")
print(pd.concat([mun["gi_clase"].value_counts().rename("sin corregir"),
                 mun["gi_clase_fdr"].value_counts().rename("con FDR")], axis=1).fillna(0).astype(int).to_string())

### 9.4 Guardado

In [ ]:
# GeoPackage con dos capas y tabla municipal sin geometría
if os.path.exists(GPKG):
    os.remove(GPKG)

mun.to_file(GPKG, layer="municipios", driver="GPKG")
dep.to_file(GPKG, layer="departamentos", driver="GPKG")
print("Guardado:", GPKG, "(capas: municipios, departamentos)")

mun.drop(columns="geometry").to_csv(TABLA_MUNICIPAL_CSV, index=False)
print("Guardado:", TABLA_MUNICIPAL_CSV)